# Fine-tuning BLIP-2 on ROCOv2 for captioning

**Step 2 of the task:** *"use the one finetuned on VQA-RAD for the first ROCO
inference run, and then further train it on a subset."* Step 1 (zero-shot) gave the
floor on the full 9,927-image test set:
BLEU-1 **0.1516** / BLEU-4 **0.0091** / METEOR **0.0984** / ROUGE-L **0.1252** /
CIDEr **0.0350** / BERTScore-F1 **0.8271**.

## The idea in one picture
Captioning is VQA with the question removed. We feed the frozen OPT decoder:

```
[ 32 soft visual vectors ]  [ "a photo of" ]  [ caption tokens ]  [ </s> ]
└─ query-only Q-Former ────┘ └── prompt ───┘  └──── the target ─────────┘
```

Labels are `-100` on the 32 soft positions **and** on the prompt, so cross-entropy
is computed **only on the caption + `</s>`**. Training on `</s>` is what un-learns the
VQA terseness: the stop token now appears after ~28 tokens instead of after 1.

## Protocol
| Split | Size | Role |
|---|---|---|
| `train` | 59,958 | fine-tune on a **subset** (N=8,000) |
| `valid` | 9,904 | **select the epoch** — never touches test |
| `test`  | 9,927 | scored **once**, at the very end |

The final test evaluation reuses the zero-shot decoding config *verbatim*
(query-only, `"a photo of"`, `num_beams=5`, `no_repeat_ngram_size=3`,
`min/max_new_tokens=8/40`) via `caption_roco.py`, so the zero-shot → fine-tuned
delta is attributable to **the weights alone**, not to a decoding change.

> **What to watch:** with a *frozen general-domain* OPT, the model may learn generic
> radiology grammar ("CT scan of the abdomen showing…") without image specificity.
> That inflates BLEU-1/ROUGE while **CIDEr stays flat** (CIDEr's TF-IDF weighting
> discounts n-grams common to every caption). A big BLEU gain with a flat CIDEr is the
> signature of *"learned the grammar, not the image."*

In [1]:
import torch
import torch.nn as nn
from PIL import Image

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
# bfloat16: fp16 makes AdamW's eps underflow -> NaN (established in the VQA run).
dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32
print(f"device = {device} | dtype = {dtype}")

device = cuda | dtype = torch.bfloat16


In [2]:
# ── Load BLIP-2 and graft the Q-Former text weights (same as every other notebook) ──
from typing import Any
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BertTokenizer

CKPT = "/home/matei/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(CKPT)
blip2 = Blip2ForConditionalGeneration.from_pretrained(CKPT, torch_dtype=dtype, low_cpu_mem_usage=True)

# The checkpoint we resume from was trained WITH the grafted text weights, so the
# architecture must match before loading -- even though captioning never uses them.
qformer_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
_QF_TEXT = torch.load("/home/matei/blip2_qformer_text_weights.pt", map_location="cpu")
qformer_word_emb = nn.Embedding.from_pretrained(_QF_TEXT["word_embeddings.weight"], freeze=False)
qformer_pos_emb  = nn.Embedding.from_pretrained(_QF_TEXT["position_embeddings.weight"], freeze=False)
qformer_text_ffn = _QF_TEXT["text_ffn"]

cfg: Any = blip2.config
print(f"d_llm={cfg.text_config.hidden_size} | d_qformer={cfg.qformer_config.hidden_size} "
      f"| num_query_tokens={cfg.num_query_tokens}")

/home/matei/miniconda3/envs/vlm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 1247/1247 [00:01<00:00, 945.36it/s] 


d_llm=2560 | d_qformer=768 | num_query_tokens=32


/tmp/ipykernel_92953/3653753196.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _QF_TEXT = torch.load("/home/matei/blip2_qformer_text_weights.pt", map_location="cpu")


## The model: one new method, `forward_caption`

`_qformer_features(pixel_values, q_ids=None)` runs the **query-only** path when no
question is given — the 32 learned queries cross-attend to the image and are projected
into OPT's embedding space as a soft visual prompt. That is standard BLIP-2 captioning.

- `generate_caption` — inference (already used for the zero-shot run).
- `forward_caption` — **new**: its training twin. Same soft prompt, but now the caption
  is appended and the language-modeling loss is taken on the caption tokens only.

Because no text enters the Q-Former, the grafted **text** feed-forward and the
word/position embeddings receive **no gradient** here. They stay dormant, and we still
save them so the checkpoint remains loadable by the same code.

In [3]:
import copy

class Blip2QFormerVQA(nn.Module):
    # Captioning = the query-only path (q_ids=None).
    def __init__(self, blip2, word_emb, pos_emb, text_ffn=None):
        super().__init__()
        self.vision_model = blip2.vision_model
        self.query_tokens = blip2.query_tokens
        self.qformer      = blip2.qformer
        for _i, _layer in enumerate(self.qformer.encoder.layer):
            if not hasattr(_layer, 'intermediate'):
                _layer.intermediate = copy.deepcopy(_layer.intermediate_query)
                _layer.output       = copy.deepcopy(_layer.output_query)
                if text_ffn is not None:
                    _layer.intermediate.load_state_dict(text_ffn[_i]['intermediate'])
                    _layer.output.load_state_dict(text_ffn[_i]['output'])
        self.language_projection = blip2.language_projection
        self.language_model      = blip2.language_model
        self.qformer_word_emb    = word_emb
        self.qformer_pos_emb     = pos_emb
        self.n_query             = blip2.config.num_query_tokens

    def _qformer_features(self, pixel_values, q_ids=None, q_att=None):
        image_embeds = self.vision_model(pixel_values).last_hidden_state
        image_atts = torch.ones(image_embeds.shape[:-1], dtype=torch.long, device=image_embeds.device)
        B = pixel_values.shape[0]
        query_tokens = self.query_tokens.expand(B, -1, -1)
        query_atts   = torch.ones(query_tokens.shape[:-1], dtype=torch.long, device=query_tokens.device)

        if q_ids is not None:                                  # VQA path (unused here)
            pos_ids  = torch.arange(q_ids.shape[1], device=q_ids.device).unsqueeze(0)
            text_emb = (self.qformer_word_emb(q_ids) + self.qformer_pos_emb(pos_ids)).to(query_tokens.dtype)
            query_embeds, attention_mask = torch.cat([query_tokens, text_emb], 1), torch.cat([query_atts, q_att], 1)
        else:                                                  # captioning: queries only
            query_embeds, attention_mask = query_tokens, query_atts

        out = self.qformer(
            query_embeds=query_embeds, query_length=self.n_query, attention_mask=attention_mask,
            encoder_hidden_states=image_embeds, encoder_attention_mask=image_atts,
        )
        query_output = out.last_hidden_state[:, :self.n_query, :].to(self.language_projection.weight.dtype)
        return self.language_projection(query_output)          # [B, 32, d_llm]

    def forward_caption(self, pixel_values, llm_ids, llm_att, labels):
        # TRAINING twin of generate_caption. LM loss on the caption tokens only.
        soft = self._qformer_features(pixel_values)                          # query-only
        soft_att = torch.ones(soft.shape[:-1], dtype=torch.long, device=soft.device)
        text_embeds    = self.language_model.get_input_embeddings()(llm_ids)
        inputs_embeds  = torch.cat([soft, text_embeds], dim=1)
        attention_mask = torch.cat([soft_att, llm_att], dim=1)
        # the 32 visual positions contribute no loss
        soft_labels = torch.full((pixel_values.shape[0], self.n_query), -100,
                                 dtype=labels.dtype, device=labels.device)
        full_labels = torch.cat([soft_labels, labels], dim=1)
        return self.language_model(inputs_embeds=inputs_embeds,
                                   attention_mask=attention_mask, labels=full_labels)

    @torch.no_grad()
    def generate_caption(self, pixel_values, prompt_ids=None, prompt_att=None, **gen_kwargs):
        soft = self._qformer_features(pixel_values)
        soft_att = torch.ones(soft.shape[:-1], dtype=torch.long, device=soft.device)
        if prompt_ids is not None:
            text_embeds    = self.language_model.get_input_embeddings()(prompt_ids)
            inputs_embeds  = torch.cat([soft, text_embeds], dim=1)
            attention_mask = torch.cat([soft_att, prompt_att], dim=1)
        else:
            inputs_embeds, attention_mask = soft, soft_att
        return self.language_model.generate(inputs_embeds=inputs_embeds,
                                            attention_mask=attention_mask, **gen_kwargs)

In [4]:
# ── Instantiate, re-inject ViT LoRA, RESUME from the VQA-RAD checkpoint ────────
from peft import LoraConfig, inject_adapter_in_model

model = Blip2QFormerVQA(blip2, qformer_word_emb, qformer_pos_emb, qformer_text_ffn)

# Recreate the SAME adapter structure the checkpoint was trained with, then load into it.
for p in model.vision_model.parameters():
    p.requires_grad = False
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules=["qkv", "projection", "fc1", "fc2"], bias="none")
inject_adapter_in_model(lora_cfg, model.vision_model)

model = model.to(device)
model.qformer_word_emb = model.qformer_word_emb.to(device, model.query_tokens.dtype)
model.qformer_pos_emb  = model.qformer_pos_emb.to(device, model.query_tokens.dtype)

RESUME_FROM = "/home/matei/vqa_checkpoints_lora/vqa_lora_final.pt"
ckpt = torch.load(RESUME_FROM, map_location=device)
model.qformer.load_state_dict(ckpt["qformer"])
model.language_projection.load_state_dict(ckpt["language_projection"])
with torch.no_grad():
    model.query_tokens.copy_(ckpt["query_tokens"].to(device, model.query_tokens.dtype))
model.qformer_word_emb.load_state_dict(ckpt["qformer_word_emb"])
model.qformer_pos_emb.load_state_dict(ckpt["qformer_pos_emb"])
model.vision_model.load_state_dict(ckpt["vit_lora"], strict=False)   # adapters only
print(f"resumed from {RESUME_FROM}")

# ── What trains: Q-Former + projection + query tokens + ViT LoRA ──
def set_trainable(m, flag):
    for p in m.parameters():
        p.requires_grad = flag

set_trainable(model.language_model, False)      # OPT frozen
set_trainable(model.qformer, True)
set_trainable(model.language_projection, True)
model.query_tokens.requires_grad = True
# the question-embedding tables are unused in captioning -> keep them frozen
model.qformer_word_emb.weight.requires_grad = False
model.qformer_pos_emb.weight.requires_grad  = False

def set_mode(train: bool):
    # Gradient checkpointing must be OFF for generation (it forces use_cache=False,
    # which makes beam search ~10x slower). Toggle it with the mode.
    if train:
        model.train()
        # use_reentrant=False is REQUIRED: the pixels carry no grad but the LoRA
        # adapters inside the ViT do, and reentrant checkpointing would drop them.
        model.vision_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.language_model.gradient_checkpointing_enable()
        model.language_model.config.use_cache = False
    else:
        model.eval()
        model.vision_model.gradient_checkpointing_disable()
        model.language_model.gradient_checkpointing_disable()
        model.language_model.config.use_cache = True

set_mode(True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable/1e6:.1f}M / total {total/1e6:.1f}M")

/tmp/ipykernel_90890/166886659.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(RESUME_FROM, map_location=device)


resumed from /home/matei/vqa_checkpoints_lora/vqa_lora_final.pt
trainable 178.5M / total 3840.0M


In [5]:
# ── ROCOv2 captioning data (dynamic padding) ─────────────────────────────────
import os, random
import pandas as pd
from torch.utils.data import Dataset as TorchDataset, DataLoader

ROCO_DIR   = "/home/matei/rocov2"
CAP_PROMPT = "a photo of"
LLM_MAXLEN = 96      # measured: 96 keeps 96.9% of captions untruncated (64 would cut 11%).
                     # Truncation strips </s> -> no stop signal -> rambling.
TRAIN_N    = 8000    # Sarah's "subset"
VAL_LOSS_N = 1000    # cheap per-epoch validation loss
VAL_GEN_N  = 500     # per-epoch generation + metrics (beam search is slow)

def load_roco(split, n=None, seed=0):
    caps = pd.read_csv(os.path.join(ROCO_DIR, f"{split}_captions.csv")).dropna(subset=["Caption"]).reset_index(drop=True)
    d = os.path.join(ROCO_DIR, split)
    recs = [{"id": r.ID, "path": os.path.join(d, f"{r.ID}.jpg"), "caption": str(r.Caption)}
            for r in caps.itertuples() if os.path.isfile(os.path.join(d, f"{r.ID}.jpg"))]
    if n is not None:
        random.Random(seed).shuffle(recs)      # fixed seed -> deterministic subset
        recs = recs[:n]
    return recs

train_recs    = load_roco("train", TRAIN_N)
val_loss_recs = load_roco("valid", VAL_LOSS_N)
val_gen_recs  = load_roco("valid", VAL_GEN_N)   # same seed -> a subset of val_loss_recs
print(f"train {len(train_recs)} | val-loss {len(val_loss_recs)} | val-gen {len(val_gen_recs)}")

PROMPT_LEN = processor.tokenizer(CAP_PROMPT, return_tensors="pt").input_ids.shape[1]  # incl. BOS
PAD_ID = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None \
         else processor.tokenizer.eos_token_id

class ROCOCaptionDataset(TorchDataset):
    def __init__(self, recs):
        self.recs = recs
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        image = Image.open(r["path"]).convert("RGB")
        pixel = processor(images=image, return_tensors="pt").pixel_values[0]  # pyright: ignore[reportCallIssue]
        full  = f"{CAP_PROMPT} {r['caption']}{processor.tokenizer.eos_token}"
        llm   = processor.tokenizer(full, truncation=True, max_length=LLM_MAXLEN, return_tensors="pt")
        ids, att = llm.input_ids[0], llm.attention_mask[0]
        labels = ids.clone()
        labels[:min(PROMPT_LEN, labels.shape[0])] = -100     # don't train on "<s> a photo of"
        return {"pixel_values": pixel, "llm_ids": ids, "llm_att": att, "labels": labels}

def collate_fn(batch):
    # pad to the longest sequence IN THIS BATCH (mean 38 tokens vs a 96 cap)
    L, n = max(b["llm_ids"].shape[0] for b in batch), len(batch)
    ids = torch.full((n, L), PAD_ID, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    lab = torch.full((n, L), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        k = b["llm_ids"].shape[0]
        ids[i, :k], att[i, :k], lab[i, :k] = b["llm_ids"], b["llm_att"], b["labels"]
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "llm_ids": ids, "llm_att": att, "labels": lab}

BATCH = 2
train_loader = DataLoader(ROCOCaptionDataset(train_recs), batch_size=BATCH, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(ROCOCaptionDataset(val_loss_recs), batch_size=BATCH, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)
_b = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in _b.items()}, f"| prompt_len={PROMPT_LEN}")

train 8000 | val-loss 1000 | val-gen 500
{'pixel_values': (2, 3, 224, 224), 'llm_ids': (2, 96), 'llm_att': (2, 96), 'labels': (2, 96)} | prompt_len=4


In [6]:
# ── Smoke test: one batch through forward_caption BEFORE committing to training ──
set_mode(True)
_b   = next(iter(train_loader))
_pix = _b["pixel_values"].to(device, dtype)
_ids, _att, _lab = _b["llm_ids"].to(device), _b["llm_att"].to(device), _b["labels"].to(device)

_out = model.forward_caption(_pix, _ids, _att, _lab)
print("forward_caption OK | loss =", float(_out.loss))
assert torch.isfinite(_out.loss), "loss is not finite"

# label masking: the prompt tokens must be ignored, the caption must not be
assert (_lab[:, :PROMPT_LEN] == -100).all(), "prompt tokens are not masked"
assert (_lab != -100).any(), "no caption tokens left to train on"
print(f"loss ignores the {model.n_query} soft positions + {PROMPT_LEN} prompt tokens")

# regression check from the VQA run: reentrant checkpointing silently drops LoRA grads
_out.loss.backward()
_named = dict(model.named_parameters())
_lora  = [n for n, p in _named.items() if "lora" in n.lower() and p.requires_grad]
_grad  = [n for n in _lora if _named[n].grad is not None]
print(f"ViT LoRA tensors receiving gradients: {len(_grad)}/{len(_lora)}")
assert len(_lora) > 0 and len(_grad) == len(_lora), "LoRA adapters received no gradient!"
model.zero_grad(set_to_none=True)
print("smoke test passed")

forward_caption OK | loss = 3.5384278297424316
loss ignores the 32 soft positions + 4 prompt tokens
ViT LoRA tensors receiving gradients: 312/312
smoke test passed


In [ ]:
# ── Validation: generation (identical config to the zero-shot run) + metrics ──
from tqdm.auto import tqdm

# IDENTICAL to caption_roco.py -- never change these, or the zero-shot comparison breaks.
GEN_KWARGS = dict(max_new_tokens=40, min_new_tokens=8, num_beams=5,
                  no_repeat_ngram_size=3, length_penalty=1.0,
                  eos_token_id=processor.tokenizer.eos_token_id,
                  pad_token_id=processor.tokenizer.eos_token_id)
CAPTION_BATCH = 8

def _load_pixels(paths):
    imgs = [Image.open(p).convert("RGB") for p in paths]
    return processor(images=imgs, return_tensors="pt").pixel_values.to(device, dtype)  # pyright: ignore[reportCallIssue]

def caption_records(recs, prompt=CAP_PROMPT, batch_size=CAPTION_BATCH):
    base = processor.tokenizer(prompt, return_tensors="pt").to(device) if prompt else None
    preds = []
    for i in tqdm(range(0, len(recs), batch_size), desc="val captioning", leave=False):
        chunk = recs[i:i+batch_size]
        pix = _load_pixels([r["path"] for r in chunk])
        B = pix.shape[0]
        pid, pat = (base.input_ids.expand(B, -1), base.attention_mask.expand(B, -1)) \
                   if base is not None else (None, None)
        gen = model.generate_caption(pix, pid, pat, **GEN_KWARGS)
        preds.extend(s.strip() for s in processor.tokenizer.batch_decode(gen, skip_special_tokens=True))
    return preds

@torch.no_grad()
def validation_loss(loader):
    set_mode(False)
    tot = n = 0
    for b in tqdm(loader, desc="val loss", leave=False):
        out = model.forward_caption(b["pixel_values"].to(device, dtype), b["llm_ids"].to(device),
                                    b["llm_att"].to(device), b["labels"].to(device))
        k = b["llm_ids"].shape[0]
        tot += float(out.loss) * k
        n += k
    return tot / n

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bertscore

def _norm(s):
    return " ".join(str(s).lower().split())

def compute_caption_metrics(preds, refs, verbose=True):
    # A partially-trained model can emit a caption that is empty after decoding;
    # BERTScore's empty-string branch then crashes. Replace empties with "." so
    # every scorer stays happy (identical guard to caption_roco.py).
    preds = [p if str(p).strip() else "." for p in preds]
    refs  = [r if str(r).strip() else "." for r in refs]
    gts = {i: [_norm(refs[i])]  for i in range(len(refs))}
    res = {i: [_norm(preds[i])] for i in range(len(preds))}
    bleu, _  = Bleu(4).compute_score(gts, res)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    meteor = sum(meteor_score([_norm(refs[i]).split()], _norm(preds[i]).split())
                 for i in range(len(preds))) / len(preds)
    _, _, F = bertscore(preds, refs, lang="en", verbose=False)
    m = {"BLEU-1": bleu[0], "BLEU-2": bleu[1], "BLEU-3": bleu[2], "BLEU-4": bleu[3],
         "METEOR": meteor, "ROUGE-L": rouge, "CIDEr": cider, "BERTScore-F1": F.mean().item()}
    if verbose:
        for k, v in m.items():
            print(f"  {k:14s}: {v:.4f}")
    return m

print("validation helpers ready")

In [ ]:
# ── Training config + a timing probe (no optimizer step -> model unchanged) ───
import time

EPOCHS, ACCUM = 3, 4          # effective batch = 2 x 4 = 8
LR, LORA_LR, WD = 1e-5, 1e-4, 0.05

set_mode(True)
_it, _n = iter(train_loader), 8
if device == "cuda": torch.cuda.synchronize()
_t0 = time.time()
for _ in range(_n):
    _b = next(_it)
    _o = model.forward_caption(_b["pixel_values"].to(device, dtype), _b["llm_ids"].to(device),
                               _b["llm_att"].to(device), _b["labels"].to(device))
    (_o.loss / ACCUM).backward()
if device == "cuda": torch.cuda.synchronize()
_dt = (time.time() - _t0) / _n
model.zero_grad(set_to_none=True)          # discard those grads: the model is untouched

_spe = len(train_loader)
print(f"{_dt:.2f} s/batch | {_spe} batches/epoch -> {_spe*_dt/60:.0f} min/epoch")
print(f"{EPOCHS} epochs ~= {EPOCHS*_spe*_dt/3600:.1f} h of training "
      f"(+ ~{EPOCHS*9} min of per-epoch validation)")

1.38 s/batch | 4000 batches/epoch -> 92 min/epoch
3 epochs ~= 4.6 h of training (+ ~27 min of per-epoch validation)

If that is too long, lower TRAIN_N (currently 8000) or EPOCHS and re-run the data cell.


In [ ]:
# ── Training loop: per-epoch checkpoint + validation loss + validation metrics ──
import os, json
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

SAVE_DIR = "/home/matei/roco_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_ckpt(tag):
    # trainable parts only; same key structure as vqa_lora_final.pt so caption_roco.py loads it
    ck = {
        "qformer":             model.qformer.state_dict(),
        "language_projection": model.language_projection.state_dict(),
        "query_tokens":        model.query_tokens.detach().cpu(),
        "qformer_word_emb":    model.qformer_word_emb.state_dict(),
        "qformer_pos_emb":     model.qformer_pos_emb.state_dict(),
        "vit_lora":            {k: v for k, v in model.vision_model.state_dict().items() if "lora" in k.lower()},
    }
    p = os.path.join(SAVE_DIR, f"roco_caption_{tag}.pt")
    torch.save(ck, p)
    return p

# three param groups: LoRA (higher LR, no decay) / matmul weights (decay) / biases+norms (no decay)
_no_decay_keys = ("bias", "norm", "query_tokens")
_lora, _decay, _no_decay = [], [], []
for _n, _p in model.named_parameters():
    if not _p.requires_grad:
        continue
    if "lora" in _n.lower():
        _lora.append(_p)
    elif _p.ndim <= 1 or any(k in _n.lower() for k in _no_decay_keys):
        _no_decay.append(_p)
    else:
        _decay.append(_p)
optimizer = AdamW([
    {"params": _decay,    "weight_decay": WD,  "lr": LR},
    {"params": _no_decay, "weight_decay": 0.0, "lr": LR},
    {"params": _lora,     "weight_decay": 0.0, "lr": LORA_LR},
])
_opt_steps = max(1, (len(train_loader) // ACCUM) * EPOCHS)
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, int(0.1 * _opt_steps)), _opt_steps)
print(f"optimizer steps: {_opt_steps} total")

history  = []
val_refs = [r["caption"] for r in val_gen_recs]

for epoch in range(EPOCHS):
    set_mode(True)
    optimizer.zero_grad(set_to_none=True)
    running = 0.0
    bar = tqdm(train_loader, desc=f"epoch {epoch+1}/{EPOCHS}")
    for step, b in enumerate(bar):
        out = model.forward_caption(b["pixel_values"].to(device, dtype), b["llm_ids"].to(device),
                                    b["llm_att"].to(device), b["labels"].to(device))
        (out.loss / ACCUM).backward()
        if (step + 1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
        running += out.loss.item()
        bar.set_postfix({"loss": out.loss.item(), "lr": scheduler.get_last_lr()[0]})
    train_loss = running / len(train_loader)

    # checkpoint FIRST -- an epoch's weights must never be lost to a scoring bug
    path = save_ckpt(f"epoch{epoch+1}")
    print(f"epoch {epoch+1} | train {train_loss:.4f} | -> {path}")
    try:
        vloss = validation_loss(val_loader)
        set_mode(False)
        vpreds = caption_records(val_gen_recs)
        # persist raw val predictions so metrics can be recomputed offline if needed
        with open(f"/home/matei/roco_val_preds_epoch{epoch+1}.json", "w") as f:
            json.dump([{"ref": r, "pred": p} for r, p in zip(val_refs, vpreds)], f, indent=1)
        vm = compute_caption_metrics(vpreds, val_refs, verbose=False)
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": vloss, "path": path, **vm})
        print(f"  val {vloss:.4f} | BERTScore {vm['BERTScore-F1']:.4f} | CIDEr {vm['CIDEr']:.4f} | "
              f"ROUGE-L {vm['ROUGE-L']:.4f} | BLEU-1 {vm['BLEU-1']:.4f}")
        print(f"  REF : {val_refs[0][:95]}")
        print(f"  PRED: {vpreds[0][:95]}")
    except Exception as e:
        import traceback; traceback.print_exc()
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "path": path, "val_error": str(e)})
        print(f"  VALIDATION FAILED (checkpoint is safe, training continues): {e}")

In [ ]:
# ── Epoch selection on VALIDATION (test is still untouched) ──────────────────
import pandas as pd

_cols = ["epoch", "train_loss", "val_loss", "BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr", "BERTScore-F1"]
print(pd.DataFrame(history).reindex(columns=_cols).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

scored = [h for h in history if "BERTScore-F1" in h]   # skip any epoch whose validation failed
best = max(scored, key=lambda r: r["BERTScore-F1"])    # ROCOv2's primary metric
print(f"\nBEST epoch by validation BERTScore: epoch {best['epoch']}")
print(f"checkpoint: {best['path']}")

# Diagnostic from the plan: did it learn the grammar, or the image?
_zs = {"BLEU-1": 0.1516, "ROUGE-L": 0.1252, "CIDEr": 0.0350, "BERTScore-F1": 0.8271}
print("\nvalidation vs. the zero-shot TEST floor (indicative -- different split):")
for k in ("BLEU-1", "ROUGE-L", "CIDEr", "BERTScore-F1"):
    print(f"  {k:14s}: {_zs[k]:.4f} -> {best[k]:.4f}   ({best[k]-_zs[k]:+.4f})")
print("\n  A large BLEU/ROUGE gain with a FLAT CIDEr means the model learned generic")
print("  caption grammar rather than image-specific content (CIDEr's TF-IDF weighting")
print("  discounts n-grams common to every caption).")

print("\n" + "=" * 78)
print("FINAL STEP -- score the selected checkpoint ONCE on test, same code path as zero-shot:")
print(f"  CKPT_PATH={best['path']} RUN_TAG=finetuned \\")
print(f"    /home/matei/miniconda3/envs/vlm/bin/python /home/matei/caption_roco.py")
print("=" * 78)